# Transaction Foundation Model on Ray — Part 2: Load & explore the data w/ Ray Data

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~15 min (most of it the one-time dataset download + normalize)


---

In Part 1, we set up the cluster. Here in Part 2, we load the benchmark dataset

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import seaborn as sns

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

## IBM TabFormer Dataset

We use the **IBM TabFormer** dataset (Padhi et al., ICASSP 2021) — the de-facto public benchmark for transaction foundation models. It has 24.4M card transactions, ~6.1k cards, 1991–2020, ~0.12% transaction-level fraud. **NVIDIA's transaction foundation model blueprint** and FATA-Trans both evaluate on it, and this series reproduces NVIDIA's result on Ray — so our numbers are directly comparable to theirs.

We build the train/val/test split the **same way NVIDIA does**, straight from the raw CSV (no precomputed files):

1. **Download** the raw `card_transaction.v1.csv` once (~2.3 GB), cached under `source/`.
2. **Temporal 80/10/10 split** by *cumulative transaction date* — the model trains on the earliest 80% of transactions, validates on the next 10%, and is tested on the most recent 10%. Splitting by time (not randomly) means the model is always evaluated on the *future*, so it can't peek ahead.
3. **100K stratified eval subsets** of val and test that preserve the natural ~0.1% fraud rate — NVIDIA's `val_eval` / `test_eval`. Scoring 100K rows instead of millions keeps evaluation cheap without distorting the metric.

NVIDIA runs this as one cuDF job on one GPU. Here it's a **Ray Data pipeline on autoscaled CPU workers** — and the output is verified identical to their reference implementation: same 19.5M rows in the same order, byte-equal eval sets (`scripts/verify_distributed_split.py`). Two things make that identity hold. A one-time conversion lands the CSV as parquet shards (a single 2.3GB CSV can't be read in parallel; no stage touches the CSV again — and a real pipeline would start from columnar files anyway). And every row carries its source position in a `__seq__` column, because a streaming engine doesn't guarantee which blocks land where — the few order-sensitive steps sort by it instead of trusting file order.

### A note on `SCALE` config

`SCALE` is the one knob that defines a run. Here it controls how many card-holders to include and the eval-subset size; in later notebooks, the sequence length, model size, and number of GPU workers. The presets (`mini` / `small` / `full`) live in `configs/`. **`mini`** keeps 200 card-holders for a fast CPU run in minutes; **`full`** uses every card and produces the Part 1 results table. Same code, different config file. Outputs are written under a per-scale path (`.../nvsplit/<scale>/`) so scales don't overwrite each other.

### Build the split from the raw CSV

The next cell is the whole split pipeline: convert the CSV once, find the temporal cutoffs from the daily counts, write the three parts with distributed filter passes, then draw the seeded eval samples. Re-runs reuse the cached output.

In [ ]:
from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale
from src.tabformer import ensure_download
from src.nvsplit import (ensure_parquet_shards, load_normalized, cutoff_dates,
                         part_filters, fresh_part_dirs, finalize_split)

SCALE = "mini"             # mini = 200 card-holders on CPU, minutes; full = every card
cfg = load_scale(SCALE)    # configs/<SCALE>.yaml — data knobs here; model/training in later parts
paths = artifact_paths(get_demo_base_dir(), SCALE)   # outputs namespaced per scale
max_users = cfg["data"]["max_users"]                 # None = every card-holder (full)

if not os.path.exists(os.path.join(paths["nvsplit"], "split_meta.json")):
    csv_path = ensure_download(paths["source"])               # one-time ~2.3GB download
    ensure_parquet_shards(csv_path, paths["source_parquet"])  # one-time CSV -> seq-tagged shards

    # NVIDIA's split protocol, distributed: the daily-count aggregation returns ~7K rows
    # to the driver (never the 24M transactions), giving the two cutoff dates; then one
    # filter+write pass per part runs across the CPU workers.
    daily = load_normalized(paths["source_parquet"], max_users) \
        .groupby("date").count().to_pandas()
    train_cutoff, test_cutoff = cutoff_dates(daily)

    dirs = fresh_part_dirs(paths["nvsplit"])
    for part, keep in part_filters(train_cutoff, test_cutoff).items():
        load_normalized(paths["source_parquet"], max_users) \
            .map_batches(keep, batch_format="pandas").write_parquet(dirs[part])

    # The seeded, order-sensitive finish (exact NVIDIA NB01 code): 100K stratified
    # val/test eval samples at the natural fraud rate + split_meta.json.
    meta = finalize_split(paths["nvsplit"], dirs, train_cutoff, test_cutoff,
                          eval_samples=cfg["data"]["eval_samples"], max_users=max_users)
else:
    meta = json.load(open(os.path.join(paths["nvsplit"], "split_meta.json")))
    print("split already built at", paths["nvsplit"])
print(json.dumps(meta, indent=2))

In [ ]:
# The train split as a Ray dataset (native TabFormer columns), plus a few derived
# analysis columns (card_id, amount as float, timestamp, is_fraud) computed per batch on
# the workers. The tokenizer in Part 3 uses the native columns directly.
from src.nvsplit import train_parquet_files
from src.tabformer import derive_explore_columns

train = ray.data.read_parquet(train_parquet_files(paths["nvsplit"])) \
                .map_batches(derive_explore_columns, batch_format="pandas")

with open(os.path.join(paths["nvsplit"], "split_meta.json")) as f:
    split_meta = json.load(f)

print(f"train split: {split_meta['train']['rows']:,} transactions  /  "
      f"{split_meta['train']['fraud_rate']*100:.3f}% fraud")
train.limit(5).to_pandas()[["User", "Card", "timestamp", "amount", "Merchant Name",
                            "MCC", "Use Chip", "Merchant State", "is_fraud"]]

## One card is one sequence

The model learns a card's transactions as a time-ordered sequence — here is what one looks like. In Part 4 the model reads these in order and learns to predict each transaction's tokens from the ones before it (next-token prediction, exactly like a language model).

In [ ]:
# One card's history, pulled from the distributed dataset (card_id 0 = User 0's first
# card): filter on the workers, collect only the ~2K matching rows.
seq = train.map_batches(lambda b: b[b["card_id"] == 0], batch_format="pandas") \
           .to_pandas().sort_values("timestamp")
print(f"card 0: {len(seq):,} transactions from "
      f"{seq['timestamp'].min().date()} to {seq['timestamp'].max().date()}")
seq[["timestamp", "amount", "Merchant Name", "MCC", "Merchant State", "is_fraud"]].head(10)

## Dataset shape and the temporal split

A final review of the data stats and temporal splits. 

In [ ]:
# Distributed aggregations — each returns a SMALL table to the driver (thousands of
# cards / hundreds of months), never the transactions themselves.
per_card = train.groupby("card_id").count().to_pandas().rename(columns={"count()": "txns"})
monthly = train.groupby("month").count().to_pandas() \
               .rename(columns={"count()": "txns"}).sort_values("month")
fraud_txns = int(train.sum("is_fraud"))
n_txns = int(per_card["txns"].sum())
fraud_rate = fraud_txns / n_txns

print(f"train transactions ...... {n_txns:,}")
print(f"cards ................... {len(per_card):,}")
print(f"txns per card ........... median {int(per_card['txns'].median()):,}  "
      f"(min {per_card['txns'].min():,}, max {per_card['txns'].max():,})")
print(f"train months ............ {monthly['month'].iloc[0]} -> {monthly['month'].iloc[-1]}")
print(f"train fraud rate ........ {fraud_rate*100:.3f}%  ({fraud_txns:,} fraudulent txns)")
print()
print("temporal split cutoffs (split_meta.json — the same split every stage uses):")
print(f"  train  <  {split_meta['train_cutoff']}")
print(f"  test   >= {split_meta['test_cutoff']}")
print(f"  eval subsets: val {split_meta['val_eval']['rows']:,} rows, "
      f"test {split_meta['test_eval']['rows']:,} rows "
      f"(stratified, ~{split_meta['test_eval']['fraud_rate']*100:.2f}% fraud)")

## Inspecting Data Distributions

Before we build out a tokenizer or a model, we inspect the data with a few plots. Each one guides an upcoming design decision. (The per-card and monthly panels come from the distributed aggregations above; the two per-transaction histograms use a ~1M-row worker-side sample.)

1. **Transactions per card.** Cards vary enormously in activity — from just a handful of transactions to nearly 50,000, with a typical (median) card around 2,500. That huge spread is what the *sequence length* has to handle: a window long enough to capture real behavioral context for active cards, but not so long that the many short-history cards (roughly a fifth have fewer than 100 transactions) become mostly padding.
2. **Transaction amount.** Amounts span several orders of magnitude — from under a dollar to a few thousand — but cluster in the tens of dollars (median ~\$33), with a right-skewed tail of larger purchases. Because amount is multiplicative rather than additive (the step from \$10 to \$100 matters like \$100 to \$1,000), a raw scalar would spend almost all its resolution on the crowded low end. So the tokenizer buckets amounts at fixed dollar thresholds (\$0/10/50/100/500/1K/5K — roughly one per order of magnitude) and treats each bucket as its own categorical token.
3. **Volume over time.** Transaction volume grows steeply through the 2000s and then levels off, so an 80/10/10 split by transaction count puts the validation and test cutoffs in the most recent stretch of calendar time. That's what we want: splitting by time trains the model on the past and evaluates it on the future, so it never gets to peek ahead.
4. **Time between transactions.** The gaps are bursty — often only a few hours apart (median ~9 hours, a fifth of them within the hour), but stretching to days or, occasionally, far longer. Ordinal position alone ("the 5th transaction") loses that. NVIDIA's tokenizer carries time as explicit fields — hour, day-of-week, and month tokens on every transaction. (It also ships an optional time-delta tokenizer that encodes the gap itself; the blueprint leaves it off, and so do we.)

In [ ]:
sns.set_theme(style="white", context="notebook")
from matplotlib.ticker import FuncFormatter
from src.tabformer import card_gap_hours

# Human-readable axis numbers: 600000 -> "600k", 2_000_000 -> "2M".
def _human(x, _):
    x = float(x)
    if abs(x) >= 1_000_000: return f"{x / 1e6:g}M"
    if abs(x) >= 1_000:     return f"{x / 1e3:g}k"
    return f"{x:g}"
human = FuncFormatter(_human)

# Per-transaction histograms only need a sample — draw ~1M rows on the workers. The
# inter-transaction gaps need each card's rows together: groupby("card_id").map_groups
# runs one card at a time across the cluster.
frac = min(1.0, 1_000_000 / max(n_txns, 1))
amt_sample = train.random_sample(frac).to_pandas()["amount"]
gap_sample = (train.groupby("card_id").map_groups(card_gap_hours, batch_format="pandas")
              .random_sample(frac).to_pandas()["gap_hours"])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) transactions per card -> motivates the sequence window length (seq_len).
ax = axes[0, 0]
sns.histplot(per_card["txns"], bins=40, color="#4C78A8", ax=ax)
ax.set_yscale("log")
ax.set_title("Transactions per card (sequence length)")
ax.set_xlabel("transactions"); ax.set_ylabel("cards (log scale)")
ax.xaxis.set_major_formatter(human); ax.yaxis.set_major_formatter(human)

# (2) amount spans orders of magnitude -> the tokenizer's fixed dollar-threshold buckets.
ax = axes[0, 1]
log_amt = np.log10(np.clip(np.abs(amt_sample.to_numpy()), 0.1, None))
sns.histplot(log_amt, bins=60, color="#F58518", ax=ax)
ax.set_title("Transaction amount (log10 $, sampled) — spans orders of magnitude")
ax.set_xlabel("log10(amount)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (3) train volume over time, with the train|val temporal cutoff (test = later, not shown).
ax = axes[1, 0]
sns.lineplot(x=pd.PeriodIndex(monthly["month"], freq="M").to_timestamp(),
             y=monthly["txns"].to_numpy(), color="#54A24B", ax=ax)
cut = pd.Timestamp(split_meta["train_cutoff"])
ax.axvline(cut, color="crimson", ls="--", lw=1)
ax.text(cut, ax.get_ylim()[1] * 0.92, "train|val", rotation=90,
        va="top", ha="right", color="crimson", fontsize=8)
ax.set_title("Train transaction volume over time (+ train|val cutoff)")
ax.set_xlabel("month"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (4) inter-transaction gaps are bursty -> why the tokenizer carries explicit time fields.
ax = axes[1, 1]
sns.histplot(np.log10(gap_sample.to_numpy()), bins=60, color="#B279A2", ax=ax)
ax.set_title("Hours between consecutive txns (log10, sampled) — bursty")
ax.set_xlabel("log10(hours since previous txn)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

sns.despine(fig=fig)
plt.tight_layout()
plt.show()

## Evaluations & how we measure performance

With only about 1 transaction in 800 fraudulent, accuracy is meaningless — a model that flags everything as valid scores 99.9% and catches nothing. We use PR-AUC (average precision) instead, which rewards ranking the rare fraud above everything else.

The eval sets keep that rarity honest: val and test are 100K-row stratified samples that *preserve* the natural ~0.1% fraud rate (~87 and ~112 frauds respectively at full scale) — NVIDIA's protocol, unchanged. Scoring 100K rows instead of 2.4M keeps evaluation cheap without changing the metric. The flip side of so few positives is that any single score is noisy, which is why Part 6 reports result distributions rather than one number.

In [ ]:
print(f"normal transactions : {(1 - fraud_rate) * 100:.3f}%")
print(f"fraud transactions  : {fraud_rate * 100:.3f}%")
print(f"imbalance ratio     : ~1 fraud per {int(round(1 / fraud_rate)):,} transactions")

## Takeaways

**Ray**
- The split is a Ray Data pipeline: filter+write passes over parquet shards on autoscaled **CPU workers**, and aggregations (`groupby(...).count()`) that return only small tables to the driver. The 19.5M-row train split is never materialized in one process, and no GPU is requested anywhere in this notebook.
- The distributed output is **verified identical** to NVIDIA's single-GPU reference — same rows, same order, byte-equal eval sets (`scripts/verify_distributed_split.py`). Row order is carried explicitly (`__seq__`) rather than trusting where a streaming engine lands blocks.
- Same code at any scale: `mini` (200 card-holders, minutes on CPU) → `full` (every card); only the config changes.

**The data**
- One card = one **time-ordered sequence** — the unit the foundation model learns over (Parts 3–4).
- **Temporal 80/10/10** split by cumulative transaction date (test = most recent), plus **100K stratified** val/test eval subsets at the natural fraud rate — identical to NVIDIA's protocol, so our numbers are comparable to theirs.
- The distributions drive the tokenizer's design in Part 3: amounts span orders of magnitude (fixed threshold buckets), time matters beyond ordinal position (hour/day-of-week/month tokens), and card histories vary enormously (windowed sequences).
- Fraud is rare (~1 in 800), so we measure with PR-AUC (average precision), not accuracy.

---

## Next

**Part 3 — Tokenize**: turn each card's native transaction rows into token sequences with NVIDIA's `FinancialTabularTokenizer` (merchant hashing + category hierarchy + temporal encoding, vocab 6251) and build the 4096-token pretraining corpus.